# Modify Hugging Face Model Weights
This notebook shows how to download a model from Hugging Face and modify all its weights, e.g., multiply them by 2.

In [1]:
%load_ext autoreload
%autoreload 2

## Step 1: Install Dependencies

In [7]:
# !pip install transformers torch

## Step 2: Load the Model

In [8]:
from transformers import AutoModel

# Load the pretrained model
model = AutoModel.from_pretrained("bert-base-uncased")

## Step 3: Modify Weights

In [9]:
import torch

def sparkler(values):
    """Sparkler quantization

    Args:
        values (tensor): torch tensor of values to be quantized

    Returns:
        tensor: quantized values
    """
    values = values.half().cpu()
    bits = values.view(torch.uint16).to(torch.int32)
    bits = (bits >> 1) << 1
    exponent_bits = (bits >> 10) & 0x1F  # 5 bits
    top3_mantissa_bits = (bits >> 7) & 0x7  # top 3 of 10 mantissa bits
    sign_bit = bits & 0x8000  # bit 15 (1 bit sign)
    
    unsparklers = torch.zeros_like(values)
    

    sparkler_indices = (exponent_bits > 11) & (exponent_bits < 20)
    unsparklers[~sparkler_indices] = values[~sparkler_indices]
    
    # Extract only values at sparkler_indices
    sparkler_exponent_bits = exponent_bits[sparkler_indices]
    sparkler_top3_mantissa_bits = top3_mantissa_bits[sparkler_indices]
    sparkler_sign_bit = sign_bit[sparkler_indices]
    
    # Reconstruct only the relevant elements
    mantissa_reconstructed = sparkler_top3_mantissa_bits << 7
    exponent_reconstructed = sparkler_exponent_bits << 10
    reconstructed_bits = sparkler_sign_bit | exponent_reconstructed | mantissa_reconstructed
    reconstructed_tensor = reconstructed_bits.to(torch.uint16).view(torch.float16)
    
    # Assign back to unsparklers
    unsparklers[sparkler_indices] = reconstructed_tensor

    return unsparklers

In [ ]:
import torch

# Put the model in evaluation mode just to avoid any training-specific behavior
model.eval()


# Modify all weights
with torch.no_grad():  # Disable gradient tracking
    for name, param in model.named_parameters():
        if param.requires_grad:
            param.data.copy_(sparkler(param.data).to(param.device))


## Step 4: (Optional) Save the Modified Model

In [ ]:
model.save_pretrained("./bert-base-uncased-sparkler")

## Step 5: (Optional) Load the Modified Model Later

In [ ]:
from transformers import AutoModel

modified_model = AutoModel.from_pretrained("./bert-base-uncased-doubled")

# Eval ViT

In [ ]:
import os
from PIL import Image
import torch
from transformers import AutoModelForImageClassification, AutoImageProcessor
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# Paths
IMAGENET_VAL_DIR = "/Users/bryansong/Data/ImageNet2012_val"

# Load processor and model
image_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
model = AutoModelForImageClassification.from_pretrained(
    "microsoft/resnet-50",
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


ResNetForImageClassification(
  (resnet): ResNetModel(
    (embedder): ResNetEmbeddings(
      (embedder): ResNetConvLayer(
        (convolution): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activation): ReLU()
      )
      (pooler): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    )
    (encoder): ResNetEncoder(
      (stages): ModuleList(
        (0): ResNetStage(
          (layers): Sequential(
            (0): ResNetBottleNeckLayer(
              (shortcut): ResNetShortCut(
                (convolution): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
                (normalization): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              )
              (layer): Sequential(
                (0): ResNetConvLayer(
                  (convolution): Conv2d(64

In [2]:
# from sparkler import sparkler

# with torch.no_grad():  # Disable gradient tracking
#     for name, param in model.named_parameters():
#         if param.requires_grad:
#             param.data.copy_(sparkler(param.data).to(param.device))

In [3]:
# Dataset and preprocessing (default huggingface transforms)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),  # Convert to [0,1]
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
])

# dataset = ImageFolder(IMAGENET_VAL_DIR, transform=transform)
from torchvision.datasets import ImageNet

dataset = ImageNet(root=IMAGENET_VAL_DIR, split="val", transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)

# Inference
correct = 0
total = 0

with torch.no_grad():
    for images, labels in tqdm(dataloader):
        images = images.to(model.device, dtype=torch.float16)
        outputs = model(pixel_values=images)
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds.cpu() == labels).sum().item()
        total += labels.size(0)
        # print("correct/total = ", correct, '/', total, '=', correct/total)

print(f"Top-1 Accuracy on ImageNet validation set: {correct / total:.4f}")

100%|██████████| 1563/1563 [03:48<00:00,  6.83it/s]

Top-1 Accuracy on ImageNet validation set: 0.7925


In [4]:
print(correct)
print(total)

39626
50000
